IMPORT MODULES

In [1]:
import math
import nltk
from nltk.tokenize import word_tokenize
from nltk import bigrams
from collections import Counter,defaultdict
from math import log
import pandas as pd

LOADING TEXT

In [2]:
df = pd.read_csv("WITH.csv",header=None,encoding="latin-1")
df.columns = ["Sentence"]

LOADING STATISTICAL MODELS

In [3]:
import spacy

nlp = spacy.load("en_core_web_sm")

COUNTERS

In [4]:
C_v,C_n,C_vp,C_np = [defaultdict(int) for _ in range(4)]

INITIALIZING COUNTERS

In [5]:
for doc in nlp.pipe(df["Sentence"]):
    for token in doc:

        #classify token as noun and verbs
        if token.pos_ == "VERB":
            C_v[token.lemma_] += 1
        elif token.pos_ == "NOUN":
            C_n[token.lemma_] += 1
        
        #if preposition found
        if token.text.lower() == "with" and token.dep_ == "prep":

            #get the association details
            head = token.head
            
            #get the object of preposition
            pobj = next((c for c in token.children if c.dep_ == "pobj"),None)

            if pobj:
                noun = pobj.lemma_
                C_np[(noun,"with")] += 1

                if head.pos_ == "VERB":
                    verb = head.lemma_
                    C_vp[(verb,"with")] += 1


HINDLE ROOTH ALGORITHM 

In [6]:
def hindle_rooth_score(v,n,p="with"):
    
    if C_v[v] == 0 or C_n[n] == 0:
        return 0
    
    #Prior probability
    p_v = C_vp[(v,p)] / C_v[v]  
    p_n = C_np[(n,p)] / C_n[n]

    # print(C_vp[(v,p)],C_v[v],C_np[(n,p)],C_n[n])

    if p_v == 0 or p_n == 0:
        return 0
    
    #log-likelihood
    return math.log2(p_v*(1-p_n)/p_n)


INPUT

In [7]:
verb = "cook"
noun = "phone"  
prep = "with"

SCORE CALCULATION

In [8]:
score = hindle_rooth_score(verb,noun,prep)

if score > 0:
    attachment = "VERB"
else:
    attachment = "NOUN"
print(f"(v={verb}, n={noun}) → {attachment}, score={score:.3f}")


(v=cook, n=phone) → NOUN, score=-1.290
